<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-08-tool-use-and-mcp/lesson-8.3-multi-tool-orchestration/notebooks/exercises-8_3.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 8.3: Multi-Tool Orchestration

*Module 8 · MCP, Tool Orchestration & Function Calling LIVE CLASS*

> Real tasks need multiple tools working together. “Find the cheapest GenAI course, calculate price with GST, and book it” requires three tools in sequence. “Get weather in 5 cities” requires five calls in parallel. This lesson builds an orchestration engine that handles parallel execution, sequential chains, conditional routing, and error recovery.

`Parallel` · `Sequential` · `Conditional` · `Orchestrator` · `60 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-8.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Parallel Execution — Run Independent Tools Simultaneously — `01_parallel.py`
2. Step 2 — Sequential Chains — Output of A Feeds Into B — `02_sequential.py`
3. Step 3 — Conditional Routing — Branch Based on Results — `03_conditional.py`


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Parallel Execution — Run Independent Tools Simultaneously

When tools don't depend on each other, run them all at once. 5x faster.

**`01_parallel.py`** · _Block 1 of 3_

PARALLEL TOOL EXECUTION


In [ ]:
# PARALLEL TOOL EXECUTION
import asyncio, time, json

# Simulated tools with artificial latency
async def get_weather(city):
    await asyncio.sleep(0.3)  # simulate API call
    temps = {"Hyderabad":34,"Mumbai":30,"Delhi":38,"Bangalore":28,"Chennai":33}
    return {"city":city, "temp":temps.get(city,25)}

async def get_exchange_rate(pair):
    await asyncio.sleep(0.2)
    rates = {"USD_INR":83.5, "EUR_INR":91.2, "GBP_INR":106.3}
    return {"pair":pair, "rate":rates.get(pair,0)}

# ── Sequential (slow) ──
async def run_sequential():
    start = time.time()
    results = []
    for city in ["Hyderabad","Mumbai","Delhi","Bangalore","Chennai"]:
        results.append(await get_weather(city))
    return results, (time.time()-start)*1000

# ── Parallel (fast) ──
async def run_parallel():
    start = time.time()
    tasks = [get_weather(c) for c in ["Hyderabad","Mumbai","Delhi","Bangalore","Chennai"]]
    results = await asyncio.gather(*tasks)
    return list(results), (time.time()-start)*1000

# ── Mixed parallel (different tool types) ──
async def run_mixed():
    start = time.time()
    results = await asyncio.gather(
        get_weather("Hyderabad"),
        get_weather("Mumbai"),
        get_exchange_rate("USD_INR"),
        get_exchange_rate("EUR_INR"),
    )
    return list(results), (time.time()-start)*1000

print("Parallel vs Sequential:\n")
seq_r, seq_t = asyncio.run(run_sequential())
par_r, par_t = asyncio.run(run_parallel())
mix_r, mix_t = asyncio.run(run_mixed())

print(f"  Sequential (5 cities): {seq_t:.0f}ms")
print(f"  Parallel   (5 cities): {par_t:.0f}ms ({seq_t/par_t:.1f}x faster)")
print(f"  Mixed      (2 weather + 2 forex): {mix_t:.0f}ms")
print(f"\n  Rule: if tools are independent, ALWAYS run parallel.")


> **What just happened?** 5 weather calls sequentially: ~1500ms (300ms each). In parallel with asyncio.gather() : ~300ms (all at once). 5x speedup. Mixed parallel runs different tool types together. Rule: if tool B doesn’t need tool A’s output, run them in parallel. Always.


### Step 2 · Sequential Chains — Output of A Feeds Into B

Search course → get price → calculate with GST → book. Each step needs the previous result.

**`02_sequential.py`** · _Block 2 of 3_

SEQUENTIAL TOOL CHAINS


In [ ]:
# SEQUENTIAL TOOL CHAINS
import json

# Tools
def search_course(topic): return {"id":"GENAI-001","name":"GenAI Complete","price":14999}
def calculate_gst(amount, rate=18): return {"subtotal":amount,"gst":round(amount*rate/100),"total":round(amount*(1+rate/100))}
def book_course(course_id, email, amount): return {"booking_id":"BK-7842","course":course_id,"amount":amount,"status":"confirmed"}

class ToolChain:
    """Execute tools in sequence, passing results forward."""
    def __init__(self): self.steps = []; self.results = []

    def add(self, func, arg_mapper):
        """arg_mapper: function that takes previous results and returns args dict."""
        self.steps.append((func, arg_mapper))
        return self

    def run(self, initial_input):
        self.results = [initial_input]
        for i, (func, mapper) in enumerate(self.steps):
            args = mapper(self.results)
            result = func(**args)
            self.results.append(result)
            print(f"    Step {i+1}: {func.__name__}({args}) → {result}")
        return self.results[-1]

# ── Build chain: search → calculate GST → book ──
chain = ToolChain()
chain.add(search_course, lambda r: {"topic": r[0]["topic"]})
chain.add(calculate_gst, lambda r: {"amount": r[1]["price"]})
chain.add(book_course, lambda r: {"course_id": r[1]["id"], "email": r[0]["email"], "amount": r[2]["total"]})

print("Sequential Chain: search → calculate → book\n")
result = chain.run({"topic":"genai", "email":"[email protected]"})
print(f"\n  Final: {result}")


### Step 3 · Conditional Routing — Branch Based on Results

If price > budget → suggest cheaper. If in stock → book. If error → retry.

**`03_conditional.py`** · _Block 3 of 3_

CONDITIONAL ROUTING — Branch based on tool results


In [ ]:
# CONDITIONAL ROUTING — Branch based on tool results
import json

def search_course(topic): return {"name":"GenAI","price":14999,"id":"G001"}
def search_cheaper(topic, max_price): return {"name":"GenAI Lite","price":7999,"id":"G002"}
def check_seat(course_id): return {"available": course_id != "FULL", "seats_left": 12}
def book(course_id, email): return {"status":"booked", "id":"BK-999"}
def add_waitlist(course_id, email): return {"status":"waitlisted", "position":3}

class ConditionalOrchestrator:
    """Route tool calls based on conditions."""
    def __init__(self): self.trace = []

    def _log(self, step, result):
        self.trace.append({"step":step, "result":result})
        print(f"    {step}: {result}")

    def enroll_student(self, topic, budget, email):
        # Step 1: Search
        course = search_course(topic)
        self._log("search", course)

        # Step 2: Budget check (CONDITIONAL)
        if course["price"] > budget:
            self._log("over_budget", f"Price {course['price']} > budget {budget}")
            course = search_cheaper(topic, budget)
            self._log("cheaper_alt", course)

        # Step 3: Availability check (CONDITIONAL)
        seat = check_seat(course["id"])
        self._log("seat_check", seat)

        if seat["available"]:
            result = book(course["id"], email)
            self._log("booked", result)
        else:
            result = add_waitlist(course["id"], email)
            self._log("waitlisted", result)

        return {"course": course["name"], "outcome": result, "steps": len(self.trace)}

print("Conditional Orchestration:\n")
orch = ConditionalOrchestrator()

print("  Scenario 1: Budget OK")
r1 = orch.enroll_student("genai", 20000, "[email protected]")
print(f"  Result: {r1}\n")

orch2 = ConditionalOrchestrator()
print("  Scenario 2: Over budget")
r2 = orch2.enroll_student("genai", 10000, "[email protected]")
print(f"  Result: {r2}")


> **What just happened?** Scenario 1 (budget 20K): search → price OK → seats available → booked. 4 steps. Scenario 2 (budget 10K): search → over budget → find cheaper alternative → seats available → booked. 5 steps. The orchestrator adapted the tool chain based on intermediate results. No pre-planned path — decisions made dynamically.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
